# Stage A - leave-one-relation-out ablation (edge-level AUPRC)

Grid: for the `complex_disease` split, train TxGNN once per **relation-ablation arm** across 5 split
seeds and record **edge-level AUPRC** (`txgnn.utils.evaluate_fb`) for indication and
contraindication. Each arm is compared against a full-KG **baseline** run over the same 5 seeds.

This notebook is standalone. It does **not** import from, read, or write
`TxGNN_PyG_Train_Eval.ipynb` or its `results/TxGNN_<split>_seed<n>_<profile>/` directories; everything
it produces lives under `results/ablation_stage_a/`.

### What an "arm" is (Option A)

Drop one relation *and its `rev_` twin* from `df_train` / `df_valid` / `df_test`, then rebuild the graph
with `create_pyg_graph(df_train, txdata.df)`. The second argument is the **untouched** full KG frame, and
`create_pyg_graph` derives `num_nodes` per type from it, so node counts stay identical across every arm -
only edges disappear.

Same-type relations (`drug_drug`, `protein_protein`, `disease_disease`, ...) get no `rev_` prefix:
`utils.reverse_rel_generation` only renames when `x_type != y_type`, so both directions already live under
the one relation name. Hetero relations (`disease_protein`, `bioprocess_protein`, ...) do have a
`rev_` twin. The drop helper filters on `{rel, 'rev_' + rel}`, which is correct in both cases.

### Two arms also change the metric-learning signature

`DistMultPredictor` builds its disease signature from `disease_etypes = ['disease_disease',
'rev_disease_protein']` (`model.py:88-89`, fallback for unseen diseases at `model.py:206-207`). For the
`disease_protein` and `disease_disease` arms the relation is therefore removed from **both** message
passing and the signature, so the ablation is consistent rather than half-applied. Those two arms are
named `drop_<rel>_full` and carry `sig_ablated: true` in their metadata; the other arms are graph-only.

### Runtime and resumability

One run is ~1 epoch of pretraining (~19 min at full KG) plus 1000 fine-tuning epochs (~5 min). Results are
written per run, immediately, to `results/ablation_stage_a/<run_key>/`. A run whose
`edge_level_test_metrics.csv` and `run_meta.json` both exist is skipped. The notebook can be re-run from
cell 1 after a crash, a kernel restart or a manual interrupt and will pick up where it stopped.

### Not in this notebook

Disease-centric evaluation (`TxEval`) - Stage A ranks relations on the cheap edge-level metric only.
Stage B (the disease-area grid) is described at the bottom but not built.

---
## 1 - Environment

In [1]:
import os, sys, json, time, gc, random, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'

# Put the PyG port FIRST on sys.path so `import txgnn` never resolves to dgl_implementation/.
sys.path.insert(0, str(PYG_ROOT))

import torch
import txgnn
import txgnn.model as txgnn_model
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, obtain_disease_profile

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# Guard for requirement 1: the Gumbel-gate switch was removed from the library in commit 2d064cc.
# Passing it (or reading m.config['use_decision_network'], or asserting on m.model.pred.block_mode)
# the way the old notebook's cell 43 did is a TypeError / KeyError now. Fail here, loudly, rather
# than eight hours into the grid.
_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit MODEL_CFG before trusting this grid.')


def set_all_seeds(seed: int):
    '''Seed every RNG the training path touches.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print('repo root     :', REPO_ROOT)
print('txgnn package :', txgnn.__file__)
print('torch         :', torch.__version__)
print('device        :', DEVICE, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())
print('model_initialize params:', sorted(_MI_PARAMS - {'self'}))
if DEVICE == 'cpu':
    print('\nWARNING: CPU only. A ~45 GPU-hour grid is not practical here.')

repo root     : /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor
txgnn package : /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor/pyg_implementation/txgnn/__init__.py
torch         : 2.6.0+cu118
device        : cuda:0 | NVIDIA GeForce RTX 4060 Ti
model_initialize params: ['agg_measure', 'attention', 'bert_measure', 'exp_lambda', 'n_hid', 'n_inp', 'n_out', 'num_walks', 'path_length', 'proto', 'proto_num', 'sim_measure', 'walk_mode']


---
## 2 - Configuration

Model and schedule are copied from the `full` profile of `TxGNN_PyG_Train_Eval.ipynb` (cell 8), which in
turn mirrors `pyg_implementation/reproduce/train.py`.

**Pretraining is one epoch.** `PRETRAIN_CFG['n_epoch'] = 1` - that one epoch is the full ~6,578-step
schedule over every training edge and is what costs ~19 min; it is not one epoch out of many. The
`smoke` profile's `n_epoch=0` is the only other value the original notebook ever uses.

The single deliberate deviation is `train_print_per_n`, raised from 100/5 to 500/100. It controls console
logging only (`TxGNN.py:191` and `TxGNN.py:253` gate a `print` and a `get_all_metrics_fb` call that
consumes no RNG and feeds nothing back into training), and 110 runs at the original setting would emit
~22,000 metric blocks. Everything that can change a number - `n_epoch`, `learning_rate`, `batch_size`,
`valid_per_n` - is untouched.

In [2]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5]          # split seeds, as in section 9 of the original notebook
PROFILE = 'full'

# verbatim from the 'full' profile / reproduce/train.py - no use_decision_network (see cell above)
MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # disease-similarity metric learning
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',   # signature = binary vector over disease + gene/protein neighbours
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_stage_a'
ABL_ROOT.mkdir(parents=True, exist_ok=True)

# --- switches ------------------------------------------------------------------------------------
RUN_GRID               = True   # section 6.2 trains; set False to inspect the plan / aggregate only
SAVE_MODELS            = False  # 110 checkpoints are large and Stage B retrains anyway
INCLUDE_TINY_RELATIONS = True   # disease_phenotype_negative - cheap, see section 3
STOP_AFTER_N_RUNS      = None   # e.g. 3 for a short unattended smoke pass over the harness

print('split   :', SPLIT)
print('seeds   :', SEEDS)
print('results :', ABL_ROOT)
print('pretrain:', PRETRAIN_CFG)
print('finetune:', FINETUNE_CFG)

split   : complex_disease
seeds   : [1, 2, 3, 4, 5]
results : /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor/results/ablation_stage_a
pretrain: {'n_epoch': 1, 'learning_rate': 0.001, 'batch_size': 1024, 'train_print_per_n': 500}
finetune: {'n_epoch': 1000, 'learning_rate': 0.0005, 'train_print_per_n': 100, 'valid_per_n': 20}


---
## 3 - The arms

Relation list from the audit's edge-count table, in descending edge count. Indication,
contraindication and off-label use are excluded - they are the task labels, not KG context.

`exposure_group` is one arm that drops all six exposure relations together (`exposure_exposure`,
`exposure_disease`, `exposure_bioprocess`, `exposure_protein`, `exposure_molfunc`,
`exposure_cellcomp`), because individually they are too small to move the metric.

`disease_phenotype_negative` is tiny; it is gated behind `INCLUDE_TINY_RELATIONS` because it costs 5 more
runs (~2 GPU-h) for a result that is almost certainly noise. Flip it off in section 2 to skip it.

In [3]:
# ordered by edge count, largest first - so an interrupted grid has covered the relations most
# likely to matter
ABLATE_RELATIONS = [
    'drug_drug',
    'anatomy_protein_present',
    'protein_protein',
    'disease_phenotype_positive',
    'bioprocess_protein',
    'bioprocess_bioprocess',
    'cellcomp_protein',
    'disease_protein',                # <- also ablates the signature
    'molfunc_protein',
    'drug_effect',
    'disease_disease',                # <- also ablates the signature
    'pathway_protein',
    'phenotype_phenotype',
    'anatomy_anatomy',
    'molfunc_molfunc',
    'drug_protein',
    'anatomy_protein_absent',
    'cellcomp_cellcomp',
    'pathway_pathway',
    'phenotype_protein',
]

EXPOSURE_GROUP = ['exposure_exposure', 'exposure_disease', 'exposure_bioprocess',
                  'exposure_protein', 'exposure_molfunc', 'exposure_cellcomp']

TINY_RELATIONS = ['disease_phenotype_negative']

# relation -> entries to strip from DistMultPredictor's disease_etypes (model.py:88, model.py:206).
# Note the asymmetry: dropping the *relation* 'disease_protein' removes the *signature entry*
# 'rev_disease_protein', because the signature reads the reverse direction (disease -> protein).
SIGNATURE_RELATIONS = {
    'disease_protein':  ['rev_disease_protein'],
    'disease_disease':  ['disease_disease'],
}


def _make_arm(name, drop, sig_drop=()):
    return {'arm': name, 'drop': list(drop), 'sig_drop': list(sig_drop),
            'sig_ablated': bool(sig_drop)}


ARMS = [_make_arm('baseline', [])]

for rel in ABLATE_RELATIONS + (TINY_RELATIONS if INCLUDE_TINY_RELATIONS else []):
    sig = SIGNATURE_RELATIONS.get(rel, ())
    # `_full` marks "graph + signature", to distinguish these two from the graph-only arms at a glance
    ARMS.append(_make_arm('drop_%s%s' % (rel, '_full' if sig else ''), [rel], sig))

ARMS.append(_make_arm('drop_exposure_group', EXPOSURE_GROUP))

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'

print('%d arms (1 baseline + %d ablations) x %d seeds = %d runs\n'
      % (len(ARMS), len(ARMS) - 1, len(SEEDS), len(ARMS) * len(SEEDS)))
for a in ARMS:
    print('  %-34s drop=%-2d %s' % (a['arm'], len(a['drop']),
                                    'signature: -' + ','.join(a['sig_drop']) if a['sig_drop'] else ''))

23 arms (1 baseline + 22 ablations) x 5 seeds = 115 runs

  baseline                           drop=0  
  drop_drug_drug                     drop=1  
  drop_anatomy_protein_present       drop=1  
  drop_protein_protein               drop=1  
  drop_disease_phenotype_positive    drop=1  
  drop_bioprocess_protein            drop=1  
  drop_bioprocess_bioprocess         drop=1  
  drop_cellcomp_protein              drop=1  
  drop_disease_protein_full          drop=1  signature: -rev_disease_protein
  drop_molfunc_protein               drop=1  
  drop_drug_effect                   drop=1  
  drop_disease_disease_full          drop=1  signature: -disease_disease
  drop_pathway_protein               drop=1  
  drop_phenotype_phenotype           drop=1  
  drop_anatomy_anatomy               drop=1  
  drop_molfunc_molfunc               drop=1  
  drop_drug_protein                  drop=1  
  drop_anatomy_protein_absent        drop=1  
  drop_cellcomp_cellcomp             drop=1  
  drop_pat

---
## 4 - Ablation mechanics

Three pieces:

1. `drop_relations(df, rels)` - filter on `{rel, 'rev_' + rel}`.
2. `AblatedData` - a stand-in for `TxData` holding the filtered frames and a rebuilt graph. `TxGNN`
   only reads `.G`, `.df`, `.df_train`, `.df_valid`, `.df_test`, `.data_folder`, `.disease_eval_idx`,
   `.split`, `.no_kg`. The full `df` is shared by reference (`create_pyg_graph` only reads it), so the
   per-seed cache is never mutated.
3. A patch on `txgnn.model.obtain_disease_profile` that drops the ablated entries from the
   `(disease_etypes, disease_nodes)` pair before building the signature. Both call sites -
   `model.py:104` in the constructor and `model.py:214` in the unseen-disease fallback - resolve the
   name from the `txgnn.model` module globals at call time, so patching the module attribute covers
   both without editing the library.

Filtering `df_valid` / `df_test` is belt-and-braces: `evaluate_graph_construct` (`utils.py:549`) iterates
`G.edge_types`, so a relation already absent from the rebuilt graph could not enter the eval graphs
anyway. Doing it keeps the three frames consistent with each other and with what gets logged.

**Why the signature patch is semantically, not numerically, load-bearing.** Once the relation is gone
from `G`, `obtain_disease_profile` already returns an all-zero block for it (`utils.py:1020-1032`: no
matching edge type -> empty `nodes` -> zeros). `sim_matrix` is cosine, and appending a common zero block
to every vector changes neither the dot products nor the norms - so the similarity matrix is identical
either way. The patch makes the intent explicit, keeps the metadata honest, and halves the signature
tensor for these two arms; it does not change the numbers. Worth knowing before designing any follow-up
that assumes "graph-only" and "graph+signature" are two distinguishable variants here - they are not.

**Edge case, not reached by this grid.** If `disease_protein` and `disease_disease` were both dropped in
one run, `keep` would be empty and the signature would have no source at all. The patch then falls back
to the unpatched call, which returns an all-zero vector for every disease; `sim_matrix`'s `eps` floor
keeps that from becoming NaN, and every disease ends up equidistant, so prototype pooling degenerates to
an unweighted mean over `proto_num` arbitrary diseases. That is a documented degenerate mode, not a
supported configuration - the two relations are separate arms here, so it never happens.

In [4]:
def drop_relations(df, rels):
    '''Remove `rels` and their rev_ twins from a split frame.'''
    victims = set()
    for r in rels:
        victims |= {r, 'rev_' + r}
    return df[~df.relation.isin(victims)].reset_index(drop=True)


class AblatedData:
    '''Minimal TxData stand-in carrying ablated splits and a rebuilt graph.'''

    def __init__(self, base, rels, data_folder, split):
        self.df = base['df']                       # untouched: fixes node counts across arms
        self.df_train = drop_relations(base['df_train'], rels)
        self.df_valid = drop_relations(base['df_valid'], rels)
        self.df_test  = drop_relations(base['df_test'],  rels)
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


# --- signature patch ------------------------------------------------------------------------------
SIG_DROP = set()          # set per run by run_one(); empty for every graph-only arm
_orig_obtain_disease_profile = obtain_disease_profile


def _obtain_disease_profile_ablated(G, disease, disease_etypes, disease_nodes):
    keep = [(e, n) for e, n in zip(disease_etypes, disease_nodes) if e not in SIG_DROP]
    if not keep:
        # degenerate mode - see the markdown above; unreachable in this grid
        return _orig_obtain_disease_profile(G, disease, disease_etypes, disease_nodes)
    etypes, nodes = zip(*keep)
    return _orig_obtain_disease_profile(G, disease, list(etypes), list(nodes))


txgnn_model.obtain_disease_profile = _obtain_disease_profile_ablated
assert txgnn_model.obtain_disease_profile is _obtain_disease_profile_ablated


# --- per-seed split cache -------------------------------------------------------------------------
_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the unablated split frames for one seed.

    Holds one seed at a time - `df` is millions of rows. prepare_split() caches to disk, so the
    first call per seed writes data/<split>_<seed>/ and later calls just read the CSVs.
    '''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()

---
## 5 - Plan

Enumerates every `run_key`, marks which are already on disk, and projects GPU-hours from the timings in
`results/TxGNN_complex_disease_seed1_full/run_config.json`.

The projection is a linear model in training-edge count: pretraining is one pass over every training edge
in batches of 1,024 (`TxGNN.py:141-148`), and the full-graph forward in fine-tuning also scales with edge
count, so both are scaled by `edges_after / edges_before`. It is a rough guide, not a promise - the
per-arm edge counts it prints are the point.

The first call to `get_seed_data` is where `prepare_split` runs. If `data/kg_directed.csv` is absent it
is rebuilt from `data/kg.csv` first, which takes 10-30 minutes once; per-seed split CSVs are cached
under `data/<split>_<seed>/` after that.

In [5]:
BASE_PRETRAIN_S, BASE_FINETUNE_S = 1155.7, 316.5   # fallback: seed-1 full run
_ref = REPO_ROOT / 'results' / ('TxGNN_%s_seed1_%s' % (SPLIT, PROFILE)) / 'run_config.json'
if _ref.exists():
    _r = json.loads(_ref.read_text())
    BASE_PRETRAIN_S = _r.get('pretrain_seconds', BASE_PRETRAIN_S)
    BASE_FINETUNE_S = _r.get('finetune_seconds', BASE_FINETUNE_S)
    print('timings from %s' % _ref.relative_to(REPO_ROOT))
print('baseline: pretrain %.1f min + finetune %.1f min = %.1f min/run\n'
      % (BASE_PRETRAIN_S / 60, BASE_FINETUNE_S / 60, (BASE_PRETRAIN_S + BASE_FINETUNE_S) / 60))

_base0   = get_seed_data(SEEDS[0])
_train   = _base0['df_train']
_counts  = _train.relation.value_counts()
N_TRAIN_EDGES = len(_train)

# every relation named in section 3 must actually exist, or the arm would silently be a second baseline
_named = (set(ABLATE_RELATIONS) | set(EXPOSURE_GROUP)
          | (set(TINY_RELATIONS) if INCLUDE_TINY_RELATIONS else set()))
_missing = sorted(r for r in _named if r not in set(_counts.index))
assert not _missing, 'relations absent from df_train (seed %d): %s' % (SEEDS[0], _missing)

rows, total_s = [], 0.0
for a in ARMS:
    victims = sorted({v for r in a['drop'] for v in (r, 'rev_' + r)} & set(_counts.index))
    dropped = int(_counts.reindex(victims).fillna(0).sum())
    frac    = (N_TRAIN_EDGES - dropped) / N_TRAIN_EDGES
    est_s   = (BASE_PRETRAIN_S + BASE_FINETUNE_S) * frac
    done    = [s for s in SEEDS if is_done(a['arm'], s)]
    todo    = [s for s in SEEDS if s not in done]
    total_s += est_s * len(todo)
    rows.append({'arm': a['arm'],
                 'relations dropped': len(victims),
                 'train edges dropped': dropped,
                 'pct of KG': 100.0 * dropped / N_TRAIN_EDGES,
                 'signature ablated': a['sig_ablated'],
                 'est min/run': est_s / 60,
                 'done': len(done), 'todo': len(todo)})

plan = pd.DataFrame(rows)
print('train edges (seed %d, full KG): %d\n' % (SEEDS[0], N_TRAIN_EDGES))
display(plan.style.format({'pct of KG': '{:.2f}', 'est min/run': '{:.1f}'}).hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]     # seed-major, matches the run order
pending  = [k for k in RUN_KEYS if not is_done(*k)]
print('\nrun_keys (%d total, %d done, %d pending):'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending)))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))
print('\nestimated remaining: %.1f GPU-hours (%.1f days unattended)'
      % (total_s / 3600, total_s / 86400))
print('order: seed-major - after seed %d finishes, every arm has n=1 and can already be ranked.'
      % SEEDS[0])

timings from results/TxGNN_complex_disease_seed1_full/run_config.json
baseline: pretrain 19.3 min + finetune 5.3 min = 24.5 min/run

[data] loading split seed 1 ...
Found local copy...
Found local copy...
Found local copy...
Found saved processed KG... Loading...
Splits detected... Loading splits....
Creating PyG graph....
Done!
[data] seed 1 ready in 6.5 s | train edges 6735310
train edges (seed 1, full KG): 6735310



arm,relations dropped,train edges dropped,pct of KG,signature ablated,est min/run,done,todo
baseline,0,0,0.00,False,24.5,1,4
drop_drug_drug,1,2221622,32.98,False,16.4,1,4
drop_anatomy_protein_present,2,2524012,37.47,False,15.3,1,4
drop_protein_protein,1,533786,7.93,False,22.6,1,4
drop_disease_phenotype_positive,2,249902,3.71,False,23.6,1,4
drop_bioprocess_protein,2,240738,3.57,False,23.7,1,4
drop_bioprocess_bioprocess,1,87924,1.31,False,24.2,1,4
drop_cellcomp_protein,2,138656,2.06,False,24.0,1,4
drop_disease_protein_full,2,133682,1.98,True,24.1,1,4
drop_molfunc_protein,2,115594,1.72,False,24.1,1,4



run_keys (115 total, 14 done, 101 pending):
  [x] baseline_seed1
  [x] drop_drug_drug_seed1
  [x] drop_anatomy_protein_present_seed1
  [x] drop_protein_protein_seed1
  [x] drop_disease_phenotype_positive_seed1
  [x] drop_bioprocess_protein_seed1
  [x] drop_bioprocess_bioprocess_seed1
  [x] drop_cellcomp_protein_seed1
  [x] drop_disease_protein_full_seed1
  [x] drop_molfunc_protein_seed1
  [x] drop_drug_effect_seed1
  [x] drop_disease_disease_full_seed1
  [x] drop_pathway_protein_seed1
  [x] drop_phenotype_phenotype_seed1
  [ ] drop_anatomy_anatomy_seed1
  [ ] drop_molfunc_molfunc_seed1
  [ ] drop_drug_protein_seed1
  [ ] drop_anatomy_protein_absent_seed1
  [ ] drop_cellcomp_cellcomp_seed1
  [ ] drop_pathway_pathway_seed1
  [ ] drop_phenotype_protein_seed1
  [ ] drop_disease_phenotype_negative_seed1
  [ ] drop_exposure_group_seed1
  [ ] baseline_seed2
  [ ] drop_drug_drug_seed2
  [ ] drop_anatomy_protein_present_seed2
  [ ] drop_protein_protein_seed2
  [ ] drop_disease_phenotype_positi

---
## 6 - The grid

`run_one` trains a single `(arm, seed)` and writes two files into `results/ablation_stage_a/<run_key>/`
the moment it finishes:

- `edge_level_test_metrics.csv` - AUROC/AUPRC for the six drug-disease edge types plus three summaries.
- `run_meta.json` - the sanity-check record: relation dropped, edge counts, `sorted(G.edge_types)` after
  ablation, the signature `disease_etypes` actually used, timings.

`MICRO (drug-disease)` is computed by `evaluate_fb` over the six DD edge types only, so it is comparable
across arms. `MACRO (all relations)` is **not** - `get_all_metrics_fb(full_mode=True)` averages over
`G.edge_types`, and an ablated graph has fewer terms in that mean. `MACRO (drug-disease)` is computed
here over the six DD relations, and is the comparable macro.

In [6]:
def run_one(arm_name, seed, verbose=True):
    '''Train one (arm, seed) and write its results. Returns the edge-level DataFrame.'''
    global SIG_DROP

    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    print('\n' + '#' * 78)
    print('# %s   seed %d   (dropping %d relation(s), signature ablated: %s)'
          % (arm_name, seed, len(arm['drop']), arm['sig_ablated']))
    print('#' * 78)

    base     = get_seed_data(seed)
    n_before = len(base['df_train'])
    victims  = sorted({v for r in arm['drop'] for v in (r, 'rev_' + r)}
                      & set(base['df_train'].relation.unique()))
    per_rel  = {r: int((base['df_train'].relation == r).sum()) for r in victims}

    set_all_seeds(seed)
    d = AblatedData(base, arm['drop'], DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)

    # requirement 5: prove the relation actually left the graph
    still_there = [r for r in victims if any(et[1] == r for et in d.G.edge_types)]
    assert not still_there, 'ablation failed, still in G: %s' % still_there

    # signature ablation - set before model_initialize, which is where the profiles are built
    SIG_DROP   = set(arm['sig_drop'])
    sig_etypes = [e for e in ['disease_disease', 'rev_disease_protein'] if e not in SIG_DROP]
    if arm['sig_ablated']:
        print('[signature] disease_etypes %s -> %s  (metric learning ablated alongside message passing)'
              % (['disease_disease', 'rev_disease_protein'], sig_etypes))

    print('[graph] train edges %d -> %d (-%d, %.2f%%) | edge types %d | dropped: %s'
          % (n_before, len(d.df_train), n_before - len(d.df_train),
             100.0 * (n_before - len(d.df_train)) / n_before, len(etypes_after), per_rel or 'none'))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**MODEL_CFG)

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations, NOT comparable across arms)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'dropped_relations_requested': arm['drop'],
        'dropped_relations_in_graph': victims,
        'dropped_edge_counts': per_rel,
        'dropped_edges_total': n_before - len(d.df_train),
        'train_edges_before': n_before,
        'train_edges_after': len(d.df_train),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'signature_ablated': arm['sig_ablated'],
        'signature_disease_etypes': sig_etypes,
        'signature_disease_etypes_default': ['disease_disease', 'rev_disease_protein'],
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    # write the metadata last - is_done() requires both files, so a crash mid-write leaves the run
    # pending rather than half-recorded
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_stage_a' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication AUPRC %.4f | contraindication AUPRC %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    # ---- free the GPU before the next run ----
    SIG_DROP = set()
    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
# Seed-major: finish every arm at seed 1 before moving to seed 2, so a partial grid is a complete
# n=1 ranking rather than a handful of fully-resolved arms. Also means one split load per seed.
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

101 run(s) pending of 115 total.

>>> 1/101  drop_anatomy_anatomy_seed1  (elapsed 0.0 h)

##############################################################################
# drop_anatomy_anatomy   seed 1   (dropping 1 relation(s), signature ablated: False)
##############################################################################
[graph] train edges 6735310 -> 6711982 (-23328, 0.35%) | edge types 49 | dropped: {'anatomy_anatomy': 23328}
Creating minibatch pretraining dataloader...
Start pre-training with #param: 994700
Epoch: 0 Step: 0 LR: 0.00100 Loss 0.6929, Pretrain Micro AUROC 0.5145 Pretrain Micro AUPRC 0.5243 Pretrain Macro AUROC 0.5567 Pretrain Macro AUPRC 0.6273
Epoch: 0 Step: 500 LR: 0.00100 Loss 0.6127, Pretrain Micro AUROC 0.7073 Pretrain Micro AUPRC 0.6737 Pretrain Macro AUROC 0.7181 Pretrain Macro AUPRC 0.7614
Epoch: 0 Step: 1000 LR: 0.00100 Loss 0.5981, Pretrain Micro AUROC 0.7289 Pretrain Micro AUPRC 0.6849 Pretrain Macro AUROC 0.7517 Pretrain Macro AUPRC 0.7649
Epoch: 

---
## 7 - Aggregate

Reads whatever is on disk - safe to run against a partial grid. Two views per relation:

- **absolute**: mean AUPRC over seeds with a 95% normal CI.
- **paired delta**: `arm - baseline` computed *per seed* and then averaged. Seeds are shared splits, so
  pairing removes split-to-split variance and is the number Stage B should rank on. An arm whose paired
  CI excludes zero is a relation the model genuinely uses.

`rank_score` is the mean of the indication and contraindication paired deltas; sorted ascending, so the
most negative (most damaging to drop, i.e. the most important relation) sits at the top.

In [ ]:
def _load_all():
    recs = []
    for meta_p in sorted(ABL_ROOT.glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'dropped_edges': meta['dropped_edges_total'],
                         'signature_ablated': meta['signature_ablated']})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    return 1.96 * x.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan


long = _load_all()
summary, wide, base = None, None, None

if long.empty:
    print('No runs on disk yet - run the grid cell in section 6 first.')
else:
    ROW = {'indication': 'drug --indication--> disease',
           'contraindication': 'drug --contraindication--> disease',
           'micro': 'MICRO (drug-disease)'}

    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    meta_by_arm = long.groupby('arm')[['dropped_edges', 'signature_ablated']].first()
    if 'baseline' in wide.index.get_level_values('arm'):
        base = wide.xs('baseline', level='arm')
    else:
        print('[warn] no baseline runs yet - deltas will be NaN.')

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'dropped_edges': int(meta_by_arm.loc[arm, 'dropped_edges']),
               'signature_ablated': bool(meta_by_arm.loc[arm, 'signature_ablated'])}
        for key in ['indication', 'contraindication', 'micro']:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == 'baseline':
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
                rec['%s_n_paired' % key] = len(g)
            elif base is not None:
                paired = (g[key] - base[key]).dropna()      # inner-joins on seed
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired) if len(paired) > 1 else np.nan
                rec['%s_n_paired' % key] = len(paired)
            else:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = np.nan, np.nan
                rec['%s_n_paired' % key] = 0
        rec['rank_score'] = np.nanmean([rec['indication_delta'], rec['contraindication_delta']])
        out.append(rec)

    summary = pd.DataFrame(out).sort_values('rank_score', na_position='last').reset_index(drop=True)
    summary.to_csv(ABL_ROOT / 'stage_a_summary.csv', index=False)

    cols = ['arm', 'n_seeds', 'dropped_edges', 'signature_ablated',
            'indication_AUPRC', 'indication_CI', 'indication_delta', 'indication_delta_CI',
            'contraindication_AUPRC', 'contraindication_CI', 'contraindication_delta',
            'contraindication_delta_CI', 'micro_AUPRC', 'micro_delta', 'rank_score']
    print('%d arms, %d runs on disk -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'stage_a_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style.format('{:.4f}', subset=cols[4:], na_rep='-').hide(axis='index'))

    incomplete = summary[summary.n_seeds < len(SEEDS)]
    if len(incomplete):
        print('\n[partial] %d arm(s) below %d seeds - deltas are provisional: %s'
              % (len(incomplete), len(SEEDS), ', '.join(incomplete.arm)))

In [ ]:
# Per-seed paired deltas for the top arms - the sign-consistency check a mean hides.
if summary is not None and base is not None:
    top = [a for a in summary.arm if a != 'baseline'][:8]
    for key in ['indication', 'contraindication']:
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - base[key]) for a in top}).T
        seed_cols = list(tbl.columns)
        tbl.columns = ['seed %d' % s for s in seed_cols]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['wins'] = (tbl[['seed %d' % s for s in seed_cols]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs baseline - %s  (negative = dropping the relation hurt)' % key)
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'wins'], na_rep='-'))

---
## 8 - What Stage B needs (not built here)

Stage A produces `results/ablation_stage_a/stage_a_summary.csv`, ranked by `rank_score` (mean paired
delta over indication and contraindication). Stage B takes the **top ~8 relations by that ranking** -
preferring arms whose paired CI excludes zero over arms that merely have a large point estimate - and
reruns those arms across the **9 disease areas** (`cell_proliferation`, `mental_health`,
`cardiovascular`, `anemia`, `adrenal_gland`, `autoimmune`, `metabolic_disorder`, `diabetes`,
`neurodigenerative`) to ask whether relation importance is area-specific.

Three things Stage B has to handle that Stage A does not:

1. **The disease-area splits are not fully zero-shot.** `utils.preprocess_kg` holds out Disease Ontology
   descendants while `utils.process_disease_area_split` filters against `data/disease_files/<area>.csv`;
   the two lists disagree, so some test diseases keep all their training drug edges. Test diseases must
   be partitioned on training drug-disease degree and only the zero-degree ones kept.
2. **Area size.** `disease_files/<area>.csv` counts ontology members, not diseases with drug edges -
   `anemia` lists 65 but only ~15 have any drug-disease edge. Check the treated-disease count per area
   before committing seeds.
3. **Metric.** Edge-level AUPRC over a handful of test diseases is noisy; Stage B likely wants the
   disease-centric protocol (`TxEval.eval_disease_centric`), which is far more expensive per run. Budget
   accordingly - 8 relations x 9 areas x seeds is a much larger grid than Stage A.

Each disease-area split also has to be built once per area before it can be trained on
(`prepare_split` writes `data/<area>_kg/` and `data/<area>_<seed>/`), a one-off cost per area.